# One log, two learners

A workshop runs one machine. Each day the operator runs it fast, runs it slowly, or replaces
it, and running wears it out until it breaks. Someone logged 100,000 days of this machine,
and on every one of them the action was picked at random. You will compute exact action
values for two policies, let Q-learning and SARSA learn from the same log, and find out
which exact table each of them learns, and what that does to the policy it hands you.

**How to work through it.** Each part asks for one or two functions. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your code is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first. `step` simulates one day, and `machine_model` writes the same rules as
arrays, `P[w, a, w2]` and the expected reward `R[w, a]`. The learners in Part 2 see only the
log and `step`, never `P` or `R`.

In [ ]:
import numpy as np

# Wear levels 0 (new) to 3 (broken). Actions: 0 = run fast, 1 = run slow, 2 = replace.
EARN_FAST = [10.0, 8.0, 5.0, 0.0]
EARN_SLOW = [6.0, 5.0, 3.0, 0.0]
WEAR_FAST, WEAR_SLOW = 0.6, 0.2       # chance that a day's running adds one wear level
REPLACE_COST = 12.0
GAMMA = 0.9
N_LEVELS, N_ACTIONS = 4, 3

def step(w, a, u):
    """One day from wear level w with action a; u is a uniform draw in [0, 1)."""
    if a == 2:
        return -REPLACE_COST, 0
    earn, wear = (EARN_FAST, WEAR_FAST) if a == 0 else (EARN_SLOW, WEAR_SLOW)
    w_next = min(w + 1, N_LEVELS - 1) if u < wear else w
    return earn[w], w_next

def machine_model():
    """The same rules as arrays: P[w, a, w'] and expected reward R[w, a]."""
    P = np.zeros((N_LEVELS, N_ACTIONS, N_LEVELS))
    R = np.zeros((N_LEVELS, N_ACTIONS))
    for w in range(N_LEVELS):
        for a, wear in ((0, WEAR_FAST), (1, WEAR_SLOW)):
            P[w, a, min(w + 1, N_LEVELS - 1)] += wear
            P[w, a, w] += 1 - wear
        R[w, 0], R[w, 1] = EARN_FAST[w], EARN_SLOW[w]
        P[w, 2, 0], R[w, 2] = 1.0, -REPLACE_COST
    return P, R

P, R = machine_model()

# The log: 100,000 days run by a policy that picks each action with probability 1/3.
# actions[t] is the action on day t (one extra entry for the day after the log),
# and us[t] is the uniform draw that decides day t's wear.
rng = np.random.default_rng(2026)
N_DAYS = 100_000
actions = rng.integers(N_ACTIONS, size=N_DAYS + 1)
us = rng.random(N_DAYS)

## Part 1 (coding task): Exact action values

Write two functions on model arrays like `P` and `R`.

- `q_star(P, R, gamma, tol=1e-10)` returns $Q^*$, of the same shape as `R`. Start from $Q = 0$
  and repeat
  $$Q(w, a) \leftarrow R[w, a] + \gamma \sum_{w'} P[w, a, w'] \max_{b} Q(w', b)$$
  for all pairs at once, until the largest change in a sweep is below `tol` (at most 10,000
  sweeps).
- `evaluate_q(P, R, pi, gamma)` returns $Q^\pi$ for a policy given as an array `pi`, where
  `pi[w, a]` is the probability of action `a` at wear level `w`. It is the solution of
  $$Q^\pi(w, a) = R[w, a] + \gamma \sum_{w'} P[w, a, w'] \sum_{b} \pi[w', b]\, Q^\pi(w', b) ,$$
  a linear system with one unknown per pair $(w, a)$. Solve it exactly with `np.linalg.solve`.

In [ ]:
def q_star(P, R, gamma, tol=1e-10):
    """Optimal action values, by repeated Bellman optimality sweeps on Q."""
    # TODO
    raise NotImplementedError


def evaluate_q(P, R, pi, gamma):
    """Exact action values of the policy pi[s, a], by one linear solve."""
    # TODO
    raise NotImplementedError

In [ ]:
# One state, two actions that both stay put: action 0 pays 1, action 1 pays 2, gamma = 0.5.
P1, R1 = np.ones((1, 2, 1)), np.array([[1.0, 2.0]])
assert np.allclose(q_star(P1, R1, 0.5), [[3, 4]]), "one-state check: Q* should be [[3, 4]], since V* = 2 / (1 - 0.5) = 4"
assert np.allclose(evaluate_q(P1, R1, np.array([[0.5, 0.5]]), 0.5), [[2.5, 3.5]]), (
    "one-state check: a 50/50 policy is worth 1.5 / (1 - 0.5) = 3, so Q should be [[1 + 0.5 * 3, 2 + 0.5 * 3]] = [[2.5, 3.5]]"
)
Q_opt = q_star(P, R, GAMMA)
gap = np.max(np.abs(R + GAMMA * P @ Q_opt.max(axis=1) - Q_opt))
assert gap < 1e-8, f"q_star does not satisfy the Bellman optimality equation: the largest gap is {gap:.2e}"
assert np.isclose(Q_opt[0, 0], 53.387230706, atol=1e-6), f"Q*[0, 0] is {Q_opt[0, 0]:.4f}, expected 53.3872"
uniform = np.full((N_LEVELS, N_ACTIONS), 1 / 3)
Q_log = evaluate_q(P, R, uniform, GAMMA)
expected_log = [[13.5625, 10.6875, -6.75], [8.1875, 6.6875, -6.75], [1.4375, 0.9375, -6.75], [-5.0625, -5.0625, -6.75]]
assert np.allclose(Q_log, expected_log, atol=1e-8), f"Q for the random policy is\n{np.round(Q_log, 4)}\nexpected\n{np.array(expected_log)}"
pi_test = np.array([[0.5, 0.3, 0.2], [0.1, 0.6, 0.3], [0.2, 0.2, 0.6], [0.0, 0.0, 1.0]])
Q_test = evaluate_q(P, R, pi_test, GAMMA)
assert np.allclose(Q_test[:, 0], [24.5698096054, 17.2081075448, 11.4483583698, 5.8326991691], atol=1e-6), (
    "wrong Q for a policy whose probabilities differ by state: the action after a move is drawn from pi "
    "at the next state s2, not at the current one"
)
print("Part 1 passed")

## Part 2 (coding task): Two learners, one log

Write `learn(rule, actions, us, gamma)`. It replays the log from wear level 0. On day `t` the
action is `a = actions[t]`, and `r, w_next = step(w, a, us[t])` gives the reward and the next
wear level. After each day it moves $Q(w, a)$ toward a target:

- With `rule == "q-learning"` the target is $r + \gamma \max_{b} Q(w_{\text{next}}, b)$.
- With `rule == "sarsa"` it is $r + \gamma\, Q(w_{\text{next}}, a_{\text{next}})$, where
  $a_{\text{next}}$ = `actions[t + 1]` is the action the log took the next day.

The step size is $1/n^{0.7}$, where $n$ counts the updates of that pair so far, this one
included. Every $Q$ starts at 0. Return the table of shape `(4, 3)`.

The check replays the first 2,000 days and compares with the exact numbers, then replays
the whole log with both rules and compares each table with the two from Part 1.

In [ ]:
def learn(rule, actions, us, gamma):
    """Replay the log with Q-learning or SARSA updates and return the learned Q."""
    # TODO
    raise NotImplementedError

In [ ]:
short_q = learn("q-learning", actions[:2001], us[:2000], GAMMA)
short_s = learn("sarsa", actions[:2001], us[:2000], GAMMA)
assert np.allclose(short_q[0], [43.1276729692, 43.3267924472, 26.7585909312], atol=1e-6), (
    f"Q-learning after 2,000 days: row 0 is {np.round(short_q[0], 4)}, expected [43.1277, 43.3268, 26.7586]. "
    "Check the target (max over the next wear level), the step size 1 / n ** 0.7 with n counting this update, "
    "and that the day's action is actions[t]"
)
assert np.allclose(short_s[0], [12.8011019704, 10.0191665486, -6.0146243432], atol=1e-6), (
    f"SARSA after 2,000 days: row 0 is {np.round(short_s[0], 4)}, expected [12.8011, 10.0192, -6.0146]. "
    "SARSA's target uses the value of actions[t + 1], the action taken on the next day, at the next wear level"
)
Q_q = learn("q-learning", actions, us, GAMMA)
Q_s = learn("sarsa", actions, us, GAMMA)
Q_opt = q_star(P, R, GAMMA)
Q_log = evaluate_q(P, R, np.full((N_LEVELS, N_ACTIONS), 1 / 3), GAMMA)
print("largest gap        to Q*    to Q of the logging policy")
for name, Q in (("Q-learning", Q_q), ("SARSA", Q_s)):
    print(f"{name:12s}  {np.abs(Q - Q_opt).max():10.2f}  {np.abs(Q - Q_log).max():10.2f}")
assert np.abs(Q_q - Q_opt).max() < 1.5, "Q-learning should end close to Q*"
assert np.abs(Q_s - Q_log).max() < 1.5, "SARSA should end close to the Q of the policy that wrote the log"
print("Part 2 passed")

## Part 3 (coding task): What each learner's policy earns

A learned table is used by acting greedily on it. Write `greedy_value(P, R, Q, gamma)`. It
builds the deterministic policy that takes `Q[s].argmax()` in each state $s$ and returns that
policy's exact value $V(s)$ for every state, using `evaluate_q`.

In [ ]:
def greedy_value(P, R, Q, gamma):
    """Exact value of acting greedily on the table Q."""
    # TODO
    raise NotImplementedError

In [ ]:
P1, R1 = np.ones((1, 2, 1)), np.array([[1.0, 2.0]])
assert np.allclose(greedy_value(P1, R1, np.array([[0.0, 9.0]]), 0.5), [4.0]), "one-state check: always taking action 1 is worth 2 / (1 - 0.5) = 4"
assert np.allclose(greedy_value(P1, R1, np.array([[9.0, 0.0]]), 0.5), [2.0]), (
    "one-state check: a table that prefers action 0 leads to a policy worth 1 / (1 - 0.5) = 2, whatever the table says"
)
V_opt = q_star(P, R, GAMMA).max(axis=1)
V_q, V_s = greedy_value(P, R, Q_q, GAMMA), greedy_value(P, R, Q_s, GAMMA)
V_log = (Q_log / 3).sum(axis=1)
print("wear   best   Q-learning's policy   SARSA's policy   logging policy")
for w in range(N_LEVELS):
    print(f"{w:4d}  {V_opt[w]:6.2f}  {V_q[w]:19.2f}  {V_s[w]:15.2f}  {V_log[w]:15.2f}")
assert np.allclose(V_q, V_opt, atol=1e-6), "acting greedily on Q-learning's table should earn exactly the optimal values here"
assert np.allclose(V_s, [31.7337036133, 19.091796875, 7.8125, 0.0], atol=1e-6), f"wrong value for SARSA's greedy policy: {np.round(V_s, 4)}"
print("Part 3 passed")

## Solution

### Part 1: Exact action values

In [ ]:
def q_star(P, R, gamma, tol=1e-10):
    """Optimal action values, by repeated Bellman optimality sweeps on Q."""
    Q = np.zeros_like(R, dtype=float)
    for _ in range(10_000):
        Q_new = R + gamma * P @ Q.max(axis=1)
        if np.max(np.abs(Q_new - Q)) < tol:
            break
        Q = Q_new
    return Q_new


def evaluate_q(P, R, pi, gamma):
    """Exact action values of the policy pi[s, a], by one linear solve."""
    n_s, n_a = R.shape
    # M[(s, a), (s2, a2)] = P[s, a, s2] * pi[s2, a2]: the chance that pair (s, a) is followed by (s2, a2)
    M = (P[:, :, :, None] * pi[None, None, :, :]).reshape(n_s * n_a, n_s * n_a)
    q = np.linalg.solve(np.eye(n_s * n_a) - gamma * M, R.reshape(-1))
    return q.reshape(n_s, n_a)

**Why it works.** `P @ Q.max(axis=1)` is the expected best value of the next state for every
pair at once, so each line of the loop is one Bellman optimality sweep on $Q$, a
$\gamma$-contraction like the one on $V$. For a fixed policy the max becomes an average with
the weights `pi[s2, :]`, and the equations become linear. Stacking the pairs gives
$q = r + \gamma M q$, where row $(s, a)$ of $M$ holds the chance of each next pair
$(s', a')$, namely $P[s, a, s']\,\pi[s', a']$. The matrix $I - \gamma M$ is invertible because
each row of $M$ sums to 1 and $\gamma < 1$.

The random policy's table shows what random play is worth: a new machine is worth only
$5.83$ on average, because a third of all days pay 12 for a replacement.

### Part 2: Two learners, one log

In [ ]:
def learn(rule, actions, us, gamma):
    """Replay the log with Q-learning or SARSA updates and return the learned Q."""
    Q = np.zeros((N_LEVELS, N_ACTIONS))
    visits = np.zeros((N_LEVELS, N_ACTIONS))
    w = 0
    for t in range(len(us)):
        a = actions[t]
        r, w_next = step(w, a, us[t])
        if rule == "q-learning":
            bootstrap = Q[w_next].max()                  # the best next action
        else:
            bootstrap = Q[w_next, actions[t + 1]]        # the next action the log took
        visits[w, a] += 1
        Q[w, a] += (r + gamma * bootstrap - Q[w, a]) / visits[w, a] ** 0.7
        w = w_next
    return Q

**Why it works.** The two rules see exactly the same days, so the only difference is the
target. SARSA's target uses the action the log took next, which was random, so its fixed
point is the Bellman expectation equation of the random policy, and it ends within about
0.6 of that table. Q-learning's target uses the best next action whatever the log did, so
its fixed point is the Bellman optimality equation, and it ends within about 0.6 of $Q^*$,
although not a single day of the log was run by a greedy policy. Every pair was tried
thousands of times, which is what Q-learning's guarantee asks of the behavior.

The step size $1/n^{0.7}$ shrinks, so the noise of single days averages out, but slowly
enough that early targets, built on poor estimates, are forgotten. With $1/n$ each pair
would weight all its updates equally, and the early ones, made when every estimate was
still near 0, would hold the values down: on this log, Q-learning with $1/n$ ends about 15
below $Q^*$.

### Part 3: What each learner's policy earns

In [ ]:
def greedy_value(P, R, Q, gamma):
    """Exact value of acting greedily on the table Q."""
    n_s, n_a = R.shape
    pi = np.zeros((n_s, n_a))
    pi[np.arange(n_s), np.asarray(Q).argmax(axis=1)] = 1.0
    return (evaluate_q(P, R, pi, gamma) * pi).sum(axis=1)     # V(s) = Q(s, pi(s))

**Why it works.** A deterministic policy is a one-hot `pi`, and its value in state $s$ is
$Q^\pi(s, \pi(s))$, the entry of `evaluate_q` at the action it takes.

Q-learning's table is close enough to $Q^*$ that its greedy policy is the optimal one: run
fast, and replace the machine once it breaks. SARSA's table ranks the actions the way the
random policy's values do, and under random play replacing a broken machine is worth
$-12 + 0.9 \times 5.83 = -6.75$, below the $-5.06$ of running it broken, because the new
machine goes back into random hands. So the greedy policy on SARSA's table never replaces
anything, and from the day the machine breaks it earns nothing. It gives up 36 at wear
level 3 and about 22 at level 0.

SARSA did what it was built to do: it learned the value of the policy that produced its
data. When the data come from some other policy and you want the best policy, you need an
off-policy target like Q-learning's.